# p109 blow-up / instability — is the ~27k event a known phenomenon?

`p109_seed485_dseed598` grokked early, sat stable ~5k–26k, then **spontaneously destabilized at ~27k** (test loss ~1e-7 → ~3) and re-cohered by ~30k. The companion notebook [`p109_late_reorganization.ipynb`](./p109_late_reorganization.ipynb) worked this event through MIScope's *own* instruments — parameter PCA, activation-DMD eigenvalues, per-neuron displacement — and found ~11 neurons exploding (one ~800× in activation magnitude) while the committed frequencies (4/14/27) survived.

This notebook asks a different question: **is this a named, well-studied phenomenon, seen from another angle?** Rather than our geometry lenses, it reaches for the optimizer- and attention-dynamics literature, where late-training spikes in a grokked model have established signatures.

Five literature threads converge on this kind of event:

1. **The slingshot mechanism** (Thilak et al., 2022). With adaptive optimizers (Adam), late training exhibits cyclic *slingshots*: the **last-layer weight norm** grows, then snaps, each snap coinciding with a training-loss spike — and grokking often begins at the first slingshot. The diagnostic is the last-layer weight norm. *(§1–§2.)*
2. **Attention entropy collapse** (Zhai et al., 2023, σReparam). A transformer instability in which attention sharpens toward one-hot — its entropy collapses — heralded by the **max (pre-softmax) attention logit** growing large. The diagnostic is the max attention logit. *(§1–§2.)*
3. **Parameter / neuron norm growth** (e.g. Merrill et al. on parameter-norm growth; outlier-feature work). Individual units' weights and activations diverge. Notebook 1's ~800× activation blow-up is the per-unit face of this. *(§2.)*
4. **Softmax Collapse / numerical instability** (Prieto et al., 2025). Once the data is fit, cross-entropy keeps dropping only by scaling logits up a fixed *naïve loss minimization* direction; pushed far enough the softmax saturates in the working float precision and the gradient is **absorbed to zero**, which can spike the loss. The diagnostics are the **logit gap** against the precision-relative saturation threshold and the **gradient-absorption fraction**. *(§4.)*
5. **Output-logit divergence** (Wortsman et al., 2023, *small-scale proxies*). Alongside attention-logit growth (their version of thread 2), they document the softmax normalizer `log Z = logsumexp(logits)` inflating without bound — the **z-loss** instability — and an AdamW-ε / update-RMS instability. The checkpoint-testable diagnostic is the `log Z` trajectory. *(§5.)*

**Working hypothesis (§1–§3):** the ~27k event is a *late, isolated slingshot* — a sudden last-layer-norm excursion — co-timed with a transient attention sharpening (a max-logit spike), with the neuron blow-up as the per-unit shadow of the same norm excursion. If so, p109's event is not exotic; it is a known instability arriving once, long after the model had apparently settled.

**First test (this notebook):** put the **last-layer weight norm** and the **max attention logit** on the same epoch axis. A slingshot predicts both spike together in the ~27k window.

---
*Conventions follow notebook 1: all data access goes through the miscope API (`variant.artifacts`, `variant.run_with_cache`), never file paths. Last-layer weight norm = Frobenius norm of the unembedding `W_U` (the final learnable map to logits); the MLP output projection `W_out` is the other candidate and is checked in the open threads. Max attention logit = max over the full (a,b) grid, all heads and query/key positions, of the pre-softmax scores at `blocks.0.attn.hook_attn_scores`.*

In [ ]:
import os
from pathlib import Path

import numpy as np
import plotly.graph_objects as go

import torch

# Locate the repo root (holds data/) and chdir there so relative data paths
# resolve from any launch directory.
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "modulo_addition_1layer").exists())
os.chdir(root)
from miscope.families.discovery import load_family_from_dir

fam = load_family_from_dir("data/modulo_addition_1layer", "data")
variant = fam.get_variant(prime=109, seed=485, data_seed=598)
PRIME = 109
GRID = [[a, b] for a in range(PRIME) for b in range(PRIME)]  # full (a,b) probe
variant

## 1. Slingshot co-diagnostic — last-layer weight norm + max attention logit

The two literature signals on one epoch axis. `W_U` is read straight from `parameter_snapshot` (cheap). The max attention logit needs a forward pass per checkpoint, so we run the full (a,b) grid through each saved model and take the maximum pre-softmax score over all heads and query/key positions.

In [ ]:
epochs = variant.artifacts.get_epochs("parameter_snapshot")


def last_layer_norm(epoch):
    """Frobenius norm of the unembedding W_U — the slingshot 'last-layer weight norm'."""
    return float(np.linalg.norm(variant.artifacts.load_epoch("parameter_snapshot", epoch)["W_U"]))


def max_attn_logit(epoch):
    """Max pre-softmax attention score over the full grid — the entropy-collapse signal."""
    _, cache = variant.run_with_cache(variant.make_probe(GRID), epoch=epoch, dtype=torch.float64)
    return float(cache["blocks.0.attn.hook_attn_scores"].max())


ll_norm = np.array([last_layer_norm(e) for e in epochs])
attn_max = np.array([max_attn_logit(e) for e in epochs])
print(f"computed {len(epochs)} epochs, {epochs[0]}..{epochs[-1]}")

In [ ]:
fig = go.Figure()
fig.add_vrect(x0=26900, x1=30000, fillcolor="orange", opacity=0.08, line_width=0,
              annotation_text="~27k event", annotation_position="top left")
fig.add_trace(go.Scatter(x=epochs, y=ll_norm, mode="lines", name="last-layer norm ||W_U||_F",
                         line=dict(color="#1f77b4")))
fig.add_trace(go.Scatter(x=epochs, y=attn_max, mode="lines", name="max attention logit",
                         line=dict(color="#d62728"), yaxis="y2"))
fig.update_layout(
    title="p109 slingshot co-diagnostic: last-layer weight norm vs max attention logit",
    xaxis_title="epoch", yaxis_title="||W_U||_F",
    yaxis2=dict(title="max attention logit", overlaying="y", side="right"),
    legend=dict(x=0.01, y=0.99), height=460)
fig.show()

In [ ]:
# Where does each signal peak, and do they co-time with the event window?
def peak(arr):
    i = int(np.argmax(arr))
    return epochs[i], float(arr[i])


e_norm, v_norm = peak(ll_norm)
e_attn, v_attn = peak(attn_max)
base = (np.array(epochs) >= 20000) & (np.array(epochs) <= 26000)  # settled-plateau reference
print(f"last-layer norm : peak {v_norm:.2f} at epoch {e_norm}  "
      f"(plateau median {np.median(ll_norm[base]):.2f}, ratio {v_norm/np.median(ll_norm[base]):.2f}x)")
print(f"max attn logit  : peak {v_attn:.2f} at epoch {e_attn}  "
      f"(plateau median {np.median(attn_max[base]):.2f}, ratio {v_attn/np.median(attn_max[base]):.2f}x)")
ev = (np.array(epochs) >= 26900) & (np.array(epochs) <= 30000)
print(f"\nin-event maxima: ||W_U|| {ll_norm[ev].max():.2f} @ {epochs[int(np.where(ev)[0][np.argmax(ll_norm[ev])])]}, "
      f"attn-logit {attn_max[ev].max():.2f} @ {epochs[int(np.where(ev)[0][np.argmax(attn_max[ev])])]}")

## 2. Localizing the diagnostics — does the blow-up surface in `W_out` and at the `=` query?

§1 showed the event is invisible to the *global* last-layer norm and the *global* max attention logit. The literature lenses are right; the **resolution** was wrong. Two localized re-runs, aimed where notebook 1 says the action is:

- **The per-neuron see-saw `||W_in[j]||` vs `||W_out[j]||`.** The ~11 exploders blow up in `W_in` (notebook 1 §1) and in activation magnitude (~800×, §7). For the function to survive — frequencies maintained, residual contribution bounded — the *output* row each exploder writes through should **down-scale** to compensate (`residual contribution = a_j · W_out[j]`). So the prediction is a see-saw: `||W_in[327]||` **up**, `||W_out[327]||` **down**, across the same window where the global `||W_out||_F` stays flat. That flat aggregate is the §1 wash-out; the per-row drop is the signal it hid.
- **Attention entropy at the `=` query position.** The cleaner entropy-collapse statement than the global max logit: at the `=` token (query index 2), how concentrated is attention over (a, b, =)? The mild, *lagging* 28100 max-logit bump from §1 may be a real small sharpening at the decision position — or noise the global max picked up elsewhere. Entropy at `=` is where to adjudicate.

In [ ]:
# Per-neuron W_in / W_out trajectories (one parameter_snapshot pass).
ps_epochs = np.array(variant.artifacts.get_epochs("parameter_snapshot"))
W_in_traj, W_out_traj = [], []
for e in ps_epochs:
    d = variant.artifacts.load_epoch("parameter_snapshot", int(e))
    W_in_traj.append(d["W_in"].T.astype(np.float64))   # (n_neurons, d_model)
    W_out_traj.append(d["W_out"].astype(np.float64))   # (n_neurons, d_model)
W_in_traj, W_out_traj = np.stack(W_in_traj), np.stack(W_out_traj)


def rank_exploders(eps, W, k=11):
    """Same definition as notebook 1 §1: peak event excursion in units of plateau drift."""
    plat, evt = (eps >= 20000) & (eps <= 26000), (eps >= 26500) & (eps <= 28500)
    ref = W[plat].mean(0)
    normal = np.linalg.norm(np.diff(W[plat], axis=0), axis=2).mean(0) + 1e-9
    peak = np.linalg.norm(W[evt] - ref[None], axis=2).max(0)
    return np.argsort(peak / normal)[::-1][:k]


exploders = rank_exploders(ps_epochs, W_in_traj)
print("exploders (W_in peak-excursion rank):", exploders.tolist())

In [ ]:
# The see-saw, for the extreme exploder n327: input norm vs output norm.
win_row = np.linalg.norm(W_in_traj, axis=2)    # (E, N)
wout_row = np.linalg.norm(W_out_traj, axis=2)  # (E, N)
wout_global = np.linalg.norm(W_out_traj, axis=(1, 2))

fig = go.Figure()
fig.add_vrect(x0=26900, x1=30000, fillcolor="orange", opacity=0.08, line_width=0)
fig.add_trace(go.Scatter(x=ps_epochs, y=win_row[:, 327], mode="lines",
                         name="||W_in[327]|| (input)", line=dict(color="#1f77b4")))
fig.add_trace(go.Scatter(x=ps_epochs, y=wout_row[:, 327], mode="lines",
                         name="||W_out[327]|| (output)", line=dict(color="#d62728"), yaxis="y2"))
fig.update_layout(title="Exploder n327 see-saw: input norm vs output norm across the event",
                  xaxis_title="epoch", xaxis_range=[24000, 31000],
                  yaxis_title="||W_in[327]||", yaxis2=dict(title="||W_out[327]||",
                  overlaying="y", side="right"), legend=dict(x=0.01, y=0.99), height=440)
fig.show()

# Cohort: did the exploders' OUTPUT rows shrink relative to plateau, vs the population?
plat = (ps_epochs >= 20000) & (ps_epochs <= 26000)
evt = (ps_epochs >= 26500) & (ps_epochs <= 30000)
out_ratio = wout_row[evt].min(0) / wout_row[plat].mean(0)   # event-min / plateau-mean, per neuron
in_ratio = win_row[evt].max(0) / win_row[plat].mean(0)      # event-max / plateau-mean, per neuron
print(f"global ||W_out||_F: plateau {wout_global[plat].mean():.2f} -> event-window "
      f"[{wout_global[evt].min():.2f}, {wout_global[evt].max():.2f}]  (flat = §1 wash-out)")
print(f"population median output-shrink ratio: {np.median(out_ratio):.2f}\n")
print("neuron   ||W_in|| grow x   ||W_out|| shrink ratio")
for j in exploders:
    print(f"  {j:>3}      {in_ratio[j]:6.1f}x          {out_ratio[j]:6.3f}")

In [ ]:
import torch


def attn_entropy_at_equals(epoch):
    """Per-head Shannon entropy of attention at the `=` query (index 2), mean over the grid."""
    _, cache = variant.run_with_cache(variant.make_probe(GRID), epoch=epoch, dtype=torch.float64)
    pat = cache["blocks.0.attn.hook_pattern"][:, :, 2, :].detach().cpu().numpy()  # (batch, heads, keys)
    ent = -(pat * np.log(np.clip(pat, 1e-12, None))).sum(-1)                       # (batch, heads)
    return ent.mean(0)                                                             # (heads,)


ent_eps = np.array([e for e in ps_epochs if 24000 <= e <= 31000])
ent = np.stack([attn_entropy_at_equals(int(e)) for e in ent_eps])  # (E, heads)

fig = go.Figure()
fig.add_vrect(x0=26900, x1=30000, fillcolor="orange", opacity=0.08, line_width=0,
              annotation_text="~27k event", annotation_position="top left")
fig.add_hline(y=float(np.log(3)), line=dict(dash="dot", color="gray"),
              annotation_text="ln 3 (uniform over a,b,=)", annotation_position="bottom right")
for h in range(ent.shape[1]):
    fig.add_trace(go.Scatter(x=ent_eps, y=ent[:, h], mode="lines", name=f"head {h}"))
fig.add_trace(go.Scatter(x=ent_eps, y=ent.mean(1), mode="lines+markers", name="mean",
                         line=dict(color="black", width=3)))
fig.update_layout(title="Attention entropy at the `=` query across the event (collapse = a dip)",
                  xaxis_title="epoch", yaxis_title="entropy (nats)", height=440)
fig.show()

base = (ent_eps >= 24000) & (ent_eps <= 26000)
mean_ent = ent.mean(1)
dip_i = int(np.argmin(mean_ent))
print(f"mean =-entropy: plateau {mean_ent[base].mean():.3f} nats -> min {mean_ent[dip_i]:.3f} "
      f"at epoch {ent_eps[dip_i]} (drop {1 - mean_ent[dip_i] / mean_ent[base].mean():.1%})")
for h in range(ent.shape[1]):
    hi = int(np.argmin(ent[:, h]))
    print(f"  head {h}: min {ent[hi, h]:.3f} at {ent_eps[hi]} (plateau {ent[base, h].mean():.3f})")

## 3. The logits across the event — does the function break and return to the *same* place?

§1–§2 read the event off the *weights* and *attention*. This pass reads it off the **output** — the logits the model actually predicts. The point of comparison is the round trip: notebook 1 found the basin "sticky in **function**, not raw coordinates" (frequencies maintained, parameters drifted). At the logit level that makes a sharp, falsifiable prediction: **the post-event logits should match the pre-event logits**, even though the weights moved — the function leaves and returns to the same place, with a genuine break in between.

Candidate checkpoints span the event (the test-loss spike peaks at ~27357, val ~3.6, but it is narrower than the 100-epoch checkpoint grid, so the highest-loss *checkpoints* sit ~27400–27600):

- **pre-event:** 20000, 22000 (clean plateau)
- **pre-event:** 26000, 26900 (instability onset)
- **broken:** 27300, 27400, 27600 (the spike + immediate aftermath)
- **recovering / post:** 28500, 29000, 30000
- **final:** 34999

For each we read the logits at the `=` query, and report cross-entropy, grid accuracy, the per-example correct-class **margin** (`logit_correct − max_other`), and the **cosine similarity of the full logit map to the pre-event reference (26000)** — the round-trip metric. Loss/accuracy are computed straight from the forward pass, not metadata, so they are self-consistent with the logits being compared.

In [ ]:
from scipy.special import logsumexp

CANDIDATES = [20000, 22000, 26000, 26900, 27300, 27400, 27600, 28500, 29000, 30000, 34999]
labels = np.array([(a + b) % PRIME for a, b in GRID])


def logits_at_equals(epoch):
    """(n_inputs, n_classes) logits read at the `=` query position."""
    logits, _ = variant.run_with_cache(variant.make_probe(GRID), epoch=epoch, dtype=torch.float64)
    return logits[:, -1, :].detach().cpu().numpy()


def readout(L, ref):
    """Cross-entropy, accuracy, per-example margin, cosine of the full logit map to ref."""
    ce = float(-(L[np.arange(len(labels)), labels] - logsumexp(L, axis=1)).mean())
    correct = L[np.arange(len(labels)), labels]
    other = L.copy(); other[np.arange(len(labels)), labels] = -np.inf
    margin = correct - other.max(1)
    cos = float((L.ravel() @ ref.ravel()) / (np.linalg.norm(L) * np.linalg.norm(ref)))
    return ce, float((L.argmax(1) == labels).mean()), margin, cos


L_ref = logits_at_equals(CANDIDATES[0])
rows = {e: readout(logits_at_equals(e), L_ref) for e in CANDIDATES}
print(f"{'epoch':>6} {'CE loss':>10} {'accuracy':>9} {'mean margin':>12}  cos-to-{CANDIDATES[0]}")
for e, (ce, acc, margin, cos) in rows.items():
    print(f"{e:>6} {ce:>10.3e} {acc:>9.3f} {margin.mean():>12.2f}    {cos:.5f}")

In [ ]:
# Per-example correct-class margin distribution per candidate epoch:
# pre tight & high -> broken collapses (negatives = misclassified) -> recovered.
fig = go.Figure()
for e in CANDIDATES:
    fig.add_trace(go.Box(y=rows[e][2], name=str(e), boxpoints=False))
fig.add_hline(y=0, line=dict(dash="dot", color="gray"),
              annotation_text="margin 0 (decision boundary)", annotation_position="right")
fig.update_layout(title="Correct-class logit margin across the event (collapse at the spike, return after)",
                  xaxis_title="epoch", yaxis_title="logit_correct - max_other", height=440,
                  showlegend=False)
fig.show()

# Round-trip: how close are the recovered logits to the pre-event function?
print(f"cosine({CANDIDATES[0]}, 30000) = {rows[30000][3]:.5f}   cosine({CANDIDATES[0]}, 34999) = {rows[34999][3]:.5f}")
worst = min(CANDIDATES, key=lambda e: rows[e][3])
print(f"least pre-like checkpoint: {worst}  (cos {rows[worst][3]:.4f}, "
      f"acc {rows[worst][1]:.3f}, CE {rows[worst][0]:.2e})")

## 4. Numerical instability — Softmax Collapse (Prieto et al., 2025)

A fourth literature thread, and the one closest to *"is the spike just an artifact?"*: **Softmax Collapse** (SC). In *Grokking at the Edge of Numerical Instability*, Prieto et al. show that once a model fits the training set, cross-entropy keeps dropping only by scaling the logits up along a fixed *naïve loss minimization* direction — ever more confident, not more correct. Pushed far enough, the softmax saturates *in the working floating-point precision*: the correct-class probability rounds to exactly 1, so the cross-entropy gradient `softmax − onehot` is **absorbed to zero**. Learning stalls along that direction and the resulting edge-of-stability dynamics can spike the loss. Their fixes are higher precision / `StableMax` and avoiding the NLM direction (`⊥Grad`).

The diagnostic is **precision-relative**. A correct example saturates when its logit gap to the competitors exceeds ≈ `ln(2/ε)` — about **16.6 nats in float32**, **36.7 nats in float64**. Which line matters depends on where the loss is computed, and **this family upcasts the logits to float64 before `log_softmax`** (`modulo_addition_1layer.py`, and every `scripts/train_*.py`), so the relevant threshold here is 36.7, not 16.6.

Two diagnostics, both off the same per-epoch `=`-query logits used in §3:

1. **Logit-gap trace (Test 1).** Track the correct-class gap (max and mean over the grid) against both precision thresholds. SC needs confidence to climb *into* the saturating regime; if the gap never reaches the float64 line, the trigger is never armed.
2. **Gradient-absorption fraction (Test 2).** Prieto's actual measurement: recompute the softmax at each precision and count the examples whose correct-class probability rounds to exactly 1 (gradient absorbed), alongside the available gradient *mass* `mean‖softmax − onehot‖₁`. SC predicts a confident plateau where gradients are absorbed, and — critically — gradient **starvation** at a loss spike. A spike driven by the *opposite* (a flood of gradient) is not SC.

**Test 3 (decisive, deferred):** a precision / `StableMax` / `⊥Grad` retrain — the natural experiment, since SC is by definition a numerical artifact. Held for a separate impact evaluation; Tests 1–2 read SC off the existing checkpoints.

---
*Note: training computes cross-entropy in float64, but the model **forward** can be run at either precision; we upcast the logits to float64 (exact) for the "true" gap, then recompute the softmax at each precision from those values — the float32 column is the counterfactual "had the loss been computed in float32," the float64 column is what this run actually trains on.*

***float64-forward control (run 2026-06-17).*** *The diagnostics above were produced with the model forward itself run in double precision (`variant.run_with_cache(..., dtype=torch.float64)`, via the dtype seam added for this experiment). Compared to the default float32 forward, every number in this section is **unchanged to ~6 significant figures** — the per-checkpoint `=`-query logits agree to ~6e-5. At this model size (d_model=128, p=109) the float32 forward is numerically faithful, so **forward precision is not a lever**: only where the loss/softmax is computed matters, and that is already float64. This rules out the late-training event being a float32 forward-pass numerical artifact, independently of the loss-side argument below.*

In [ ]:
# Tests 1+2 share one forward pass per epoch: read the =-query logits, derive the
# logit gap (Test 1) and the precision-relative gradient absorption (Test 2). The
# softmax normalizer log Z is recorded too — reused by §5 (Wortsman output-logit divergence).
SC_FP32 = float(np.log(2 / np.finfo(np.float32).eps))  # ~16.6 nats: float32 saturation gap
SC_FP64 = float(np.log(2 / np.finfo(np.float64).eps))  # ~36.7 nats: float64 saturation gap


def sc_diagnostics(z, y):
    """SC quantities at one checkpoint. z: (N, C) float64 logits; y: (N,) labels."""
    rows = np.arange(len(y))
    other = z.copy(); other[rows, y] = -np.inf
    gap = z[rows, y] - other.max(1)                              # correct-class margin per example
    lz = logsumexp(z, axis=1)                                   # log Z = output-logit normalizer (§5)
    one_minus_p = -np.expm1(z[rows, y] - lz)                    # accurate 1 - p_correct (grad size)

    def absorbed_frac(dtype):
        """Fraction of grid whose softmax saturates (p_correct rounds to 1) in `dtype`."""
        zz = z.astype(dtype); zz = zz - zz.max(1, keepdims=True)
        e = np.exp(zz); sm = e / e.sum(1, keepdims=True)
        return float(np.mean(sm[rows, y] == dtype(1.0)))

    return dict(max_abs_logit=float(np.abs(z).max()),
                gap_max=float(gap.max()), gap_mean=float(gap.mean()),
                grad_mass=float(2.0 * one_minus_p.mean()),      # mean L1 of (softmax - onehot)
                logZ_mean=float(lz.mean()), logZ_max=float(lz.max()),
                absorbed_fp32=absorbed_frac(np.float32), absorbed_fp64=absorbed_frac(np.float64))


sc_keys = ["max_abs_logit", "gap_max", "gap_mean", "grad_mass",
           "logZ_mean", "logZ_max", "absorbed_fp32", "absorbed_fp64"]
sc = {k: [] for k in sc_keys}
for e in epochs:  # one forward pass per checkpoint, like §1
    d = sc_diagnostics(logits_at_equals(int(e)).astype(np.float64), labels)
    for k in sc_keys:
        sc[k].append(d[k])
sc = {k: np.array(v) for k, v in sc.items()}
sc_ep = np.array(epochs)

print(f"float32 SC gap = {SC_FP32:.1f} nats   float64 SC gap = {SC_FP64:.1f} nats   "
      f"(cross-entropy is computed in float64)\n")
print(f"{'epoch':>6} {'max|z|':>8} {'gap_max':>8} {'gap_mean':>9} {'grad mass':>11} {'absorb32':>9} {'absorb64':>9}")
for e in CANDIDATES:
    i = int(np.where(sc_ep == e)[0][0])
    print(f"{e:>6} {sc['max_abs_logit'][i]:>8.1f} {sc['gap_max'][i]:>8.2f} {sc['gap_mean'][i]:>9.2f} "
          f"{sc['grad_mass'][i]:>11.2e} {sc['absorbed_fp32'][i]:>9.3f} {sc['absorbed_fp64'][i]:>9.3f}")

In [ ]:
# Test 1 — does confidence ever climb into the saturating regime?
fig = go.Figure()
fig.add_vrect(x0=26900, x1=30000, fillcolor="orange", opacity=0.08, line_width=0,
              annotation_text="~27k event", annotation_position="top left")
fig.add_hline(y=SC_FP64, line=dict(dash="dash", color="#d62728"),
              annotation_text="float64 SC gap (36.7)", annotation_position="top right")
fig.add_hline(y=SC_FP32, line=dict(dash="dot", color="gray"),
              annotation_text="float32 SC gap (16.6)", annotation_position="bottom right")
fig.add_trace(go.Scatter(x=sc_ep, y=sc["gap_max"], mode="lines", name="gap (most-confident example)",
                         line=dict(color="#1f77b4")))
fig.add_trace(go.Scatter(x=sc_ep, y=sc["gap_mean"], mode="lines", name="gap (grid mean)",
                         line=dict(color="#1f77b4", dash="dot")))
fig.update_layout(title="Test 1 — correct-class logit gap vs softmax-saturation thresholds",
                  xaxis_title="epoch", yaxis_title="logit gap to competitors (nats)",
                  legend=dict(x=0.01, y=0.99), height=440)
fig.show()

over = "below" if sc["gap_max"].max() < SC_FP64 else "above"
print(f"gap_max over all training = {sc['gap_max'].max():.2f} nats "
      f"(float64 SC needs {SC_FP64:.1f}) — peak confidence stays {over} the float64 line")
print(f"max |logit| ever = {sc['max_abs_logit'].max():.1f} "
      f"(float64 exp overflows near 709 — no overflow regime)")

In [ ]:
# Test 2 — gradient absorption per precision, and the gradient mass that survives.
plat = (sc_ep >= 20000) & (sc_ep <= 26000)
evt = (sc_ep >= 26900) & (sc_ep <= 30000)

fig = go.Figure()
fig.add_vrect(x0=26900, x1=30000, fillcolor="orange", opacity=0.08, line_width=0,
              annotation_text="~27k event", annotation_position="top left")
fig.add_trace(go.Scatter(x=sc_ep, y=sc["absorbed_fp32"], mode="lines",
                         name="absorbed fraction (float32 counterfactual)", line=dict(color="gray")))
fig.add_trace(go.Scatter(x=sc_ep, y=sc["absorbed_fp64"], mode="lines",
                         name="absorbed fraction (float64, training)", line=dict(color="#d62728")))
fig.add_trace(go.Scatter(x=sc_ep, y=sc["grad_mass"], mode="lines", name="gradient mass (float64)",
                         line=dict(color="#2ca02c"), yaxis="y2"))
fig.update_layout(title="Test 2 — softmax-gradient absorption and surviving gradient mass",
                  xaxis_title="epoch", yaxis_title="fraction of grid with gradient absorbed",
                  yaxis2=dict(title="mean ||softmax - onehot||_1", overlaying="y", side="right",
                              type="log"),
                  legend=dict(x=0.5, y=-0.2, orientation="h"), height=440)
fig.show()

print(f"absorbed float64: max over ALL training = {sc['absorbed_fp64'].max():.3f}  "
      f"(SC never bites in the precision actually used)")
print(f"absorbed float32: plateau median {np.median(sc['absorbed_fp32'][plat]):.3f}  "
      f"-> event min {sc['absorbed_fp32'][evt].min():.3f}  "
      f"(SC-vulnerable in float32; the event moves AWAY from the edge)")
print(f"gradient mass: plateau median {np.median(sc['grad_mass'][plat]):.2e}  "
      f"-> event max {sc['grad_mass'][evt].max():.2e}  "
      f"({sc['grad_mass'][evt].max() / np.median(sc['grad_mass'][plat]):.0f}x — a flood, not starvation)")

### §4 reading — Softmax Collapse is ruled out (in the precision actually trained)

**Test 1 (logit gap).** Across all of training the correct-class gap tops out at **35.25 nats** — it never reaches the **float64** saturation line at 36.7. Even the single most-confident example at its most-confident epoch stays ~1.5 nats short of where the float64 softmax would round to one-hot. Max `|logit|` peaks at ~148, far below the float64 `exp` overflow (~709). The SC trigger is never armed in the working precision.

**Test 2 (gradient absorption).** The float64 absorbed fraction is **0.000 at every epoch** — no example's gradient is ever absorbed in the precision the loss is actually computed in. The float32 counterfactual is the interesting part: on the plateau **~62%** of the grid *would* be SC-saturated *if the loss were computed in float32*. This run lives right at the **float32 edge of numerical instability** — exactly Prieto's regime — and the codebase's float64-loss convention is the silent mitigation keeping it clear.

**The event moves the wrong way for SC.** At the spike the gap *collapses* (17 → 6 nats, §3's margin collapse), the float32 absorbed fraction *recedes to 0*, and the gradient mass *spikes ~5×10⁴* (plateau ~3e-7 → ~1e-2 in the event window). Softmax Collapse is gradient **starvation** from runaway over-confidence; the ~27k event is its antithesis — a confidence *collapse* that floods the model with gradient.

**Verdict.** The ~27k event is **not** Softmax Collapse, and it is not a numerical artifact of the trained precision. A **float64-forward control** (running the model forward, not just the loss, in double precision) leaves every number in this section unchanged to ~6 significant figures — direct confirmation that the *forward* precision, not only the loss precision, is not the lever. What the tests *do* surface is a latent fragility: in float32 this exact run would sit ~62% inside the SC regime on its plateau — why the float64-loss convention earns its keep, and the cleanest motivation for the deferred **Test 3** (a float32-vs-float64 / `StableMax` / `⊥Grad` retrain) if we ever want to see whether the *plateau itself*, not the event, changes shape under precision.

## 5. Output-logit divergence — z-loss instability (Wortsman et al., 2023)

*Small-scale proxies for large-scale Transformer training instabilities* reproduces two large-scale instabilities in small models. One is **attention-logit growth → entropy collapse** (their qk-layernorm fix) — the same mechanism as Zhai, already put to rest in §1–§2. The other is new to this notebook: **output-logit divergence**, where the softmax normalizer `log Z = logsumexp(logits)` drifts upward without bound as the logits inflate, and their fix is **z-loss**, an auxiliary `λ·log²Z` penalty that pins `log Z` near 0.

This is the **output-scale companion to §4**. SC (§4) asks whether the logits got confident enough to saturate the softmax in the working precision; z-loss asks whether the logit *scale itself* is running away. Same naïve-loss-minimization driver, different failure mode. This family trains **without z-loss**, so `log Z` is free to drift — the test is simply whether it does, and whether the 27k event is a `log Z` divergence or (as §3–§4 suggest) yet another transient *dip*.

A third Wortsman thread — the **AdamW-ε / update-RMS** instability — needs the optimizer's second-moment state, which `parameter_snapshot` doesn't save; it is a retrain-only test, filed with the deferred Test 3.

In [ ]:
# Wortsman 'output logit divergence': does the softmax normalizer log Z = logsumexp(logits)
# run away? z-loss exists to pin it near 0. (Reuses §4's per-epoch pass — log Z is in `sc`.)
fig = go.Figure()
fig.add_vrect(x0=26900, x1=30000, fillcolor="orange", opacity=0.08, line_width=0,
              annotation_text="~27k event", annotation_position="top left")
fig.add_hline(y=0.0, line=dict(dash="dot", color="gray"),
              annotation_text="z-loss target (log Z ~ 0)", annotation_position="bottom right")
fig.add_trace(go.Scatter(x=sc_ep, y=sc["logZ_mean"], mode="lines", name="log Z (grid mean)",
                         line=dict(color="#1f77b4")))
fig.add_trace(go.Scatter(x=sc_ep, y=sc["logZ_max"], mode="lines", name="log Z (grid max)",
                         line=dict(color="#1f77b4", dash="dot")))
fig.update_layout(title="§5 — output-logit normalizer log Z across training (divergence = unbounded growth)",
                  xaxis_title="epoch", yaxis_title="log Z = logsumexp(logits at =)",
                  legend=dict(x=0.01, y=0.99), height=440)
fig.show()

plat = (sc_ep >= 20000) & (sc_ep <= 26000)
evt = (sc_ep >= 26900) & (sc_ep <= 30000)
i_pk = int(np.argmax(sc["logZ_mean"]))
print(f"log Z (mean): plateau median {np.median(sc['logZ_mean'][plat]):.1f}, final(34999) {sc['logZ_mean'][-1]:.1f}  "
      f"-> flat = no post-grok divergence (z-loss target ~0)")
print(f"log Z global max {sc['logZ_mean'].max():.1f} at epoch {sc_ep[i_pk]}  "
      f"({'during grokking, not late' if sc_ep[i_pk] < 6000 else 'late'})")
print(f"event: log Z dips to {sc['logZ_mean'][evt].min():.1f} (confidence collapse) then recovers "
      f"— a trough, not a blow-up")

### §5 reading — output-logit divergence is ruled out too

`log Z` climbs during grokking (≈19 → ≈89 across epochs 200–5000), then **pins flat at ≈87.5 from 5k through 35k** — its global maximum (92.9) sits at epoch **4600**, *inside* the grokking transition, not late. There is **no post-grok runaway**: final-epoch `log Z` (87.7) equals the plateau (87.5). And the 27k event is, once more, a **trough not a divergence** — `log Z` *dips* to 33.7 at 27600 (the §3/§4 confidence collapse) and recovers. Wortsman's output-logit-divergence instability does not describe this event.

**The §4 echo.** Note where p109 *lives*: `log Z ≈ 87` with no z-loss is exactly the inflated-logit regime z-loss exists to suppress — and **weight decay (`wd = 1.0`) is what bounds it** (flat, not diverging), just as float64 bounds the SC edge in §4. Two papers' instabilities, two different recipe choices (weight decay, float64) silently holding this model clear of both. The pattern is consistent: p109 sits *inside* the regime each instability warns about, and a standing recipe constraint — not luck — keeps the instability from biting.

**Not testable here.** Wortsman's third thread — the AdamW-ε / update-RMS instability — needs the optimizer's second-moment state, which `parameter_snapshot` does not save. It is a retrain-only test, filed with the deferred Test 3.

## Reading & open threads

Across all five resolutions — global weights/attention (§1), localized weights/attention (§2), the output logits (§3), the numerical-precision regime (§4), and the output-logit scale (§5) — the ~27k event matches **none** of the named instabilities: not the slingshot signature, not attention-entropy collapse, not Softmax Collapse, not output-logit divergence. The literature angle does not name this event; what it does is let us rule those mechanisms out cleanly and pin what the event actually is: an **uncompensated input/activation blow-up in a few MLP rows** that briefly softens the output margin and then **round-trips back to the same function**.

**§1 — global aggregates are blind.** `||W_U||_F` peaks at epoch 2200 (grokking) and is flat (~11.5) at 27k; the global max attention logit peaks at epoch 200 (init), with only a mild ~1.3× bump at 28100, *after* the spike. The ~800× blow-up is invisible to both.

**§2 — the localized re-runs falsify the two natural mechanisms:**

- **The see-saw is one-sided — `W_out` does *not* compensate.** Exploder input rows blow up (n327 **43.5×**, n400 22×, cohort 8–44×), but their output rows do **not** shrink to offset it: per-row shrink ratios are **0.93–1.26** (exploders at/above 1.0) vs a population median of 0.82. So `a_j·W_out[j]` **overshoots** — which *is* the loss spike. **This corrects notebook 1 §7's conjecture** that `W_out` must down-scale: it doesn't; recovery comes from the input/activation blow-up **relaxing**, not output rescaling.
- **No attention-entropy collapse.** `=`-query entropy moves 0.620 → 0.601 nats (3%); the strongest head dips ~11% at 28300 — mild, head-specific, and *lagging* the spike.
- **`W_out` is the event-sensitive "last layer," `W_U` is not** — global `||W_out||_F` excurses (11.74 → [10.1, 14.8]) where `||W_U||_F` stayed flat, but it *grows* transiently rather than slingshot-snapping.

**§3 — the output round-trips to the same function:**

- **Function leaves and returns.** Cosine of the full logit map to the pre-event reference (26000) dips to **0.982** at the most-perturbed checkpoint (27400) and recovers to **0.998** (30000) / **0.9995** (34999). The model relaxes back to essentially the *same* logits despite the parameter drift — direct output-level confirmation of notebook 1's "basin sticky in **function**, not coordinates."
- **Accuracy never drops at any saved checkpoint** (1.000 throughout, including the broken 27400/27600); the break surfaces only as a **margin collapse** (mean correct-class margin 17.07 → **6.16** at 27600). The true loss-3.6 misclassification peak at **27357 falls *between* checkpoints** — the 100-epoch grid undersamples it. Saved-checkpoint *accuracy* alone would miss the event entirely: a **time-resolution** analog of §1's global→local wash-out (the signal needs finer time sampling, just as §2 needed finer unit/head sampling).

**§4 — not a numerical artifact, but a latent float32 fragility surfaces.** Training computes cross-entropy in **float64**, where the softmax saturates only past a **36.7-nat** logit gap; the correct-class gap tops out at **35.25** across all of training and the float64 gradient-absorbed fraction is **0.000 everywhere** — the SC trigger is never armed in the precision actually used. The event moves the *wrong* way for SC: the gap *collapses* (17 → 6 nats) and the gradient mass *spikes ~5×10⁴* — a gradient flood, the antithesis of SC's gradient starvation. The byproduct finding: on the plateau **~62%** of the grid *would* be SC-saturated in **float32** — this run sits squarely at Prieto's edge of numerical instability, and the float64-loss convention is the silent mitigation holding it clear.

**§5 — output-logit divergence is ruled out, and the same "inside the regime, bounded by recipe" pattern repeats.** `log Z = logsumexp(logits)` climbs during grokking (≈19 → ≈89) and then **pins flat at ≈87.5 from 5k through 35k** — global max 92.9 at epoch 4600 (*inside* grokking), final-epoch 87.7 = plateau. No post-grok runaway; the event is a **trough** (`log Z` dips to 33.7 at 27600) not a divergence. As in §4, p109 *lives* in the regime the instability warns about — `log Z ≈ 87` with no z-loss is exactly what z-loss exists to suppress — but **weight decay (`wd = 1.0`) bounds it**, the §4-echo: a standing recipe constraint, not luck, keeps it clear.

**Net read.** The ~27k event is a transient, self-healing **confidence collapse**: a handful of MLP input rows blow up, the output layer does not cancel the surplus, the correct-class margin briefly softens (instantaneously enough to spike the loss to ~3.6, but never flipping a prediction at saved resolution), and the whole configuration then relaxes back to the same function (cosine ~0.9995). Not a slingshot, not entropy collapse, not Softmax Collapse, not output-logit divergence — uncompensated, localized, reversible, and clear of both the float64 saturation regime and any `log Z` runaway throughout. A consistent meta-pattern across §4–§5: p109 sits *inside* the high-logit regime two instability papers warn about, held stable by standing recipe choices (float64 loss, weight decay) rather than by staying clear of the regime.

**Still open:**

- **Catch the true peak.** The loss-3.6 spike at 27357 has no checkpoint. If re-trained (no resume path — REQ_149), dense checkpoints across 27300–27400 would show whether accuracy *does* break at the instantaneous peak, or whether even the worst instant is margin-only.
- **Timing the overshoot.** Put `||W_in[j]||`, the contribution `a_j·W_out[j]`, the margin (§3), and the loss on one axis — does the contribution peak coincide with the margin trough?
- **Per-head max logit on the exploder-routing head**, paired with §2's per-head entropy, to finish the attention-side localization.
- **Precision & optimizer state as variables (deferred Test 3).** The plateau sits ~62% inside the float32 SC regime and at `log Z ≈ 87`; a float32-vs-float64 / `StableMax` / z-loss / `⊥Grad` retrain — with the optimizer's second-moment state logged, to reach Wortsman's untested AdamW-ε / update-RMS thread — would show whether the *plateau and the event itself* change shape under precision and regularization. The decisive test that §4–§5's instabilities are absent, pending an impact evaluation.
- **Control on p113** (clean grokker): expect flat `W_U`/max-logit, a flat see-saw, no `=`-entropy dip, a flat logit cosine ~1.0, and a flat `log Z` throughout, plus the same float64-clear / float32-edge SC profile — confirming every signal here is event-specific, and that the float32 / high-`log Z` fragility is a property of the *solution*, not this event.
- **Promote to an analyzer?** Several of these are cheap per-epoch scalars off one `=`-query forward pass: the see-saw pair (`||W_in[j]||`, `||W_out[j]||`), `=`-entropy, the logit-cosine round-trip, the §4 logit-gap / absorption-fraction, and the §5 `log Z`. The output-side group (gap, absorption, `log Z`, margin, cosine) is a natural single **output-logit health** analyzer; the see-saw and the logit-cosine are the two that *did* localize/round-trip the event. Candidates beside the §7 `neuron_activation_spectrum` sketch.